In [1]:
import pandas as pd
from core import utils, paths, database
from datetime import datetime
import os
import sys
import pandas as pd
from core import data_processing as dp

In [2]:
from core import database as db

In [3]:
from core.paths import ASSETS_PATH, ASSETS_TIME_TABLE 

In [9]:
pd.options.display.max_columns = None

In [ ]:
#query builder
def query_builder(period_key):
    """
    Adott period_key alapján SQL lekérdezést épít.
    A period_key lehet: 'wtd_set', 'ptd_set', 'ytd_set'.
    Feltételezi, hogy a db.* függvények elérhetők:
      - db.get_current_tesco_week()
      - db.get_current_tesco_period()
      - db.get_fyear_start_period()
    """

    # A SELECT/GROUP BY oszlopok igazítva a CTE-hez (nap -> t.nap)
    period_sets = {
        'wtd_set': {
            'cols': 't.nap AS day',
            'interval_template': "dmtm_fw_code = '{current_week}'",
            'group': 't.nap'
        },
        'ptd_set': {
            'cols': 't.dmtm_fw_code AS fiscal_week',
            'interval_template': "dmtm_fp_code = '{current_period}'",
            'group': 't.dmtm_fw_code'
        },
        'ytd_set': {
            'cols': 't.dmtm_fp_code AS fiscal_period',
            'interval_template': "dmtm_fp_code BETWEEN '{start_period}' AND '{end_period}'",
            'group': 't.dmtm_fp_code'
        }
    }

    if period_key not in period_sets:
        raise ValueError("Invalid period key. Választható: 'wtd_set', 'ptd_set', 'ytd_set'.")

    # --- Dátum/period változók előállítása a period_key alapján ---
    current_week = None
    current_period = None
    start_period = None
    end_period = None

    match period_key:
        case 'wtd_set':
            current_week = db.get_current_tesco_week()
        case 'ptd_set':
            current_period = db.get_current_tesco_period()
        case 'ytd_set':
            start_period = db.get_fyear_start_period()
            end_period = db.get_current_tesco_period()
        case _:
            raise ValueError(f"Ismeretlen period_key: {period_key}")

    # --- Az intervallum feltétel sablonjának formázása ---
    interval_template = period_sets[period_key]['interval_template']

    if period_key == 'wtd_set':
        if current_week is None:
            raise ValueError("current_week nincs beállítva.")
        interval_sql = interval_template.format(current_week=current_week)

    elif period_key == 'ptd_set':
        if current_period is None:
            raise ValueError("current_period nincs beállítva.")
        interval_sql = interval_template.format(current_period=current_period)

    else:  # 'ytd_set'
        if start_period is None or end_period is None:
            raise ValueError("start_period / end_period nincs beállítva.")
        interval_sql = interval_template.format(start_period=start_period, end_period=end_period)

    # --- SELECT/GROUP BY mezők ---
    cols = period_sets[period_key]['cols']
    group = period_sets[period_key]['group']

    # --- SQL összeállítása ---
    query = f"""
    SELECT
        rep.cntr_code AS `country`,
        {cols},
        rep.dmat_div_code AS `division`,
        rep.dmat_div_des AS `division_name`,
        rep.dmat_dep_code AS `department`,
        rep.dmat_dep_des AS `department_name`,
        rep.dmat_sec_code AS `section`,
        rep.dmat_sec_des AS `section_name`,
        rep.dmat_grp_code AS `group`,
        rep.dmat_grp_des AS `group_name`,

        rc.rc_cat_name AS rpc_category,

        /* --- Alap metrikák (TY/LY) --- */
        COALESCE(SUM(CASE WHEN t.ev = 'ty' THEN sales.slsms_salex_cs / NULLIF(rate.dmexr_rate, 0) ELSE 0 END), 0) AS `sales_excl_vat_gbp_ty`,
        COALESCE(SUM(CASE WHEN t.ev = 'ty' THEN sales.slsms_margin   / NULLIF(rate.dmexr_rate, 0) ELSE 0 END), 0) AS `scan_margin_gbp_ty`,
        COALESCE(SUM(CASE WHEN t.ev = 'ty' THEN sales.slsms_unit_cs                              ELSE 0 END), 0) AS `sold_unit_ty`,

        COALESCE(SUM(CASE WHEN t.ev = 'ly' THEN sales.slsms_salex_cs / NULLIF(rate.dmexr_rate, 0) ELSE 0 END), 0) AS `sales_excl_vat_gbp_ly`,
        COALESCE(SUM(CASE WHEN t.ev = 'ly' THEN sales.slsms_margin   / NULLIF(rate.dmexr_rate, 0) ELSE 0 END), 0) AS `scan_margin_gbp_ly`,
        COALESCE(SUM(CASE WHEN t.ev = 'ly' THEN sales.slsms_unit_cs                              ELSE 0 END), 0) AS `sold_unit_ly`,

        /* --- LFL metrikák (TY/LY) --- */
        COALESCE(SUM(CASE WHEN t.ev = 'ty' AND sales.slsms_lfl_flag = 1
                            THEN sales.slsms_salex_cs / NULLIF(rate.dmexr_rate, 0) ELSE 0 END), 0) AS `sales_excl_vat_gbp_lfl`,
        COALESCE(SUM(CASE WHEN t.ev = 'ty' AND sales.slsms_lfl_flag = 1
                            THEN sales.slsms_margin   / NULLIF(rate.dmexr_rate, 0) ELSE 0 END), 0) AS `scan_margin_gbp_lfl`,
        COALESCE(SUM(CASE WHEN t.ev = 'ty' AND sales.slsms_lfl_flag = 1
                            THEN sales.slsms_unit_cs                              ELSE 0 END), 0) AS `sold_unit_lfl`,

        COALESCE(SUM(CASE WHEN t.ev = 'ly' AND sales.slsms_lfl_flag = 1
                            THEN sales.slsms_salex_cs / NULLIF(rate.dmexr_rate, 0) ELSE 0 END), 0) AS `sales_excl_vat_gbp_lflb`,
        COALESCE(SUM(CASE WHEN t.ev = 'ly' AND sales.slsms_lfl_flag = 1
                            THEN sales.slsms_margin   / NULLIF(rate.dmexr_rate, 0) ELSE 0 END), 0) AS `scan_margin_gbp_lflb`,
        COALESCE(SUM(CASE WHEN t.ev = 'ly' AND sales.slsms_lfl_flag = 1
                            THEN sales.slsms_unit_cs                              ELSE 0 END), 0) AS `sold_unit_lflb`,

        /* --- 2Y LFL metrikák (opcionális) --- */
        COALESCE(SUM(CASE WHEN t.ev = 'ty' AND sales.slsms_2ylfl_flag = 1
                            THEN sales.slsms_salex_cs / NULLIF(rate.dmexr_rate, 0) ELSE 0 END), 0) AS `sales_excl_vat_gbp_ty_2ylfl`,
        COALESCE(SUM(CASE WHEN t.ev = 'ty' AND sales.slsms_2ylfl_flag = 1
                            THEN sales.slsms_margin   / NULLIF(rate.dmexr_rate, 0) ELSE 0 END), 0) AS `scan_margin_gbp_ty_2ylfl`,
        COALESCE(SUM(CASE WHEN t.ev = 'ty' AND sales.slsms_2ylfl_flag = 1
                            THEN sales.slsms_unit_cs                              ELSE 0 END), 0) AS `sold_unit_ty_2ylfl`,

        COALESCE(SUM(CASE WHEN t.ev = 'ly' AND sales.slsms_2ylfl_flag = 1
                            THEN sales.slsms_salex_cs / NULLIF(rate.dmexr_rate, 0) ELSE 0 END), 0) AS `sales_excl_vat_gbp_2ylflb`,
        COALESCE(SUM(CASE WHEN t.ev = 'ly' AND sales.slsms_2ylfl_flag = 1
                            THEN sales.slsms_margin   / NULLIF(rate.dmexr_rate, 0) ELSE 0 END), 0) AS `scan_margin_gbp_2ylflb`,
        COALESCE(SUM(CASE WHEN t.ev = 'ly' AND sales.slsms_2ylfl_flag = 1
                            THEN sales.slsms_unit_cs                              ELSE 0 END), 0) AS `sold_unit_2ylflb`

    FROM dm.dim_artrep_details rep

    JOIN dw.sl_sms sales
      ON sales.slsms_dmat_id = rep.slad_dmat_id
     AND sales.slsms_cntr_id IN (1, 2, 4)
     AND sales.slsms_cntr_id = rep.cntr_id

    /* Opcionális store join:
    JOIN dm.dim_stores store
      ON sales.slsms_dmst_id = store.dmst_store_id
     AND sales.slsms_cntr_id = store.cntr_id
    */

    JOIN (
        SELECT
            'ty' AS ev,
            dmtm_fw_weeknum AS het,
            dmtm_d_code     AS nap,
            dmtm_fw_code,
            dmtm_fp_code
        FROM dm.dim_time_d
        WHERE {interval_sql}

        UNION
        SELECT
            'ly' AS ev,
            dmtm_fw_weeknum       AS het,
            dmtm_d_code_ly_offset AS nap,
            dmtm_fw_code,
            dmtm_fp_code
        FROM dm.dim_time_d
        WHERE {interval_sql}
    ) t
      ON t.nap = sales.part_col

    JOIN (
        SELECT dmexr_cntr_id, dmexr_rate
        FROM dw.dim_exchange_rates
        WHERE dmexr_dmtm_fy_code = (SELECT fiscal_year FROM tesco_analysts.pmajor1_fiscal_year_for_filter)
          AND dmexr_crncy_to = 'GBP'
          AND dmexr_cntr_id IN (1, 2, 4)
    ) rate
      ON sales.slsms_cntr_id = rate.dmexr_cntr_id

    JOIN (
        SELECT dmrrc_cntr_id, dmrrc_code_id, rc_cat_name
        FROM dw.dim_retail_rc
        JOIN dm.dim_rc_ret_category ON rc_cat_id = dmrrc_rc_cat_id
        GROUP BY dmrrc_cntr_id, dmrrc_code_id, rc_cat_name
    ) rc
      ON sales.slsms_cntr_id = rc.dmrrc_cntr_id
     AND sales.slsms_nrc     = rc.dmrrc_code_id

    WHERE rep.cntr_code IN ('HU','CZ','SK')
    and rep.dmat_div_code = 0004

    GROUP BY
        {group},
        rep.cntr_code,
        rep.dmat_div_code,
        rep.dmat_div_des,
        rep.dmat_dep_code,
        rep.dmat_dep_des,
        rep.dmat_sec_code,
        rep.dmat_sec_des,
        rep.dmat_grp_code,
        rep.dmat_grp_des,
        rc.rc_cat_name;
    """
    return query


In [4]:
path = r"\\euprgvmfs01.tesco-europe.com\GMforCentralEurope\Supply Chain\OOB\Open Order Book 2026.02.09.xlsm"

In [5]:
df= pd.read_excel(path,sheet_name="Detail", skiprows=4)

In [ ]:
2273678
2273679
2273680
2273713
2276758
2276772
2277242
2277247
2277262
2277264
2277352
2277353
2277357
2290352
2273678
2273679
2273680
2273713
2276758
2276772
2277242
2277247
2277262
2277264
2277352
2277353
2277357
2290352


In [10]:
df[df["ORDER NO"] == 2273678]

,ORDER NO,LM PO NO,SUPPLIER,ORIG APPROVAL DATE,NOT BEFORE DATE,NOT AFTER DATE,ITEM,Master TPN,Vendor Order Number\n(Retek PO),ITEM_DESC,LM_TPND,Ordered on SPK,Cases on pallet,NO of pallet,DIVISION,GROUP_NO,DEPT,CLASS,SUBCLASS,DIV_NAME,GROUP_NAME,DEPT_NAME,CLASS_NAME,SUB_NAME,LOCATION,QTY ORDERED,QTY RECEIVED,CURRENCY CODE,UNIT COST,SUPP PACK SIZE,IS/ DOMESTIC,SUPPLIER NAME,ORIGINAL DELIVERY WEEK,NEW DELIVERY WEEK,h,PERIOD NUMBER,H1_H2,SC COMMENT,COUNTRY,QTY OUTSTANDING (Units),VALUE OUTSTANDING (EUR),VALUE OUTSTANDING (GBP),CASES ON ORDER,Past Due Order,HDC SOH UNITS,HDC SOH CASES,HDC WKS COVER,ETA Port of Arrival,Port of Arrival,CNTR no,cc,IN TRANSIT INFO,DETAILS,STATUS,raised delivery week vs.actual delivery week\n[in weeks],ETA HDC,ETA HDC WEEK,CONFIRMED BOOKING week,CONFIRMED BOOKING day,Order waiting for ASN,STOCK IN TRANSIT,Hierarchy from,Reporting dept,Reporting Sec,Reporting Group,3rd party storage,HDC_Advices,ELC_price_EUR,ELC_price_GBP,ELC_value_GBP,Hazardous
137,2273678,999-55384,4000001,2025.08.21,2026.02.02,2026.02.02,2005100964736,2005100641153,999-55384,SPK Tesco Recycled Cotton Terry tea towel 5pk ...,2269177,yes,45.0,2.133333,35,195,681,515,65,Centralized Hardline,CH Home,CH Home,CH Dining,CH Kitchen textiles,79005,288,0,USD,2.1,3,IS FOB,TESCO INTL SOURCING LTD (USD)USD,NaN,202550,NaN,2025_12,H2,NaN,Centralised,288,544.864864,459.486292,96.0,N,0,0.0,0.0,2026-02-03,GALANTA - SLOVAK REPUBLIC,CARU9779501,0.0,DEPARTED LOAD PORT,Departed Origin - INTUT - 03-Dec-2025 - Not Bo...,SHIPPED & NOT BOOKED,NaN,2026-10-02,NaN,202631.0,2026-10-02 00:00:00,NaN,202544,3519568151565,Home,Dining,Kitchen textiles,NaN,2026-10-02,0.4193,0.353597,101.835947,NaN
